# P2. Build a Production RAG Pipeline (Capstone)

**Tier:** Projects
**Estimated time:** 60 minutes
**Prerequisites:** 14, 15, 24, 26, 33
**Priority:** 🟡 Important — consolidation: the RAG skills already exist across Tier 3 and `07_rag_learning`, but wiring them together WITH evals and a regression gate is what makes a RAG system production-grade rather than a demo. *If skipped, revisit when:* before building your first production RAG system — use this notebook as the checklist.
**Source material:** Notebooks 14/15 (RAG fundamentals, vector DBs), `07_rag_learning/` series, notebook 24 (evals), notebook 26 (LLM-as-judge), notebook 33 (CI for AI)

## What You'll Learn
- Assembling `ragkit`'s pipeline (chunk → embed → store → retrieve → generate) into one production-shaped function
- Retrieval evals: hit-rate@k and MRR against a golden set of (question, expected source document) pairs
- Faithfulness judging: scoring whether a generated answer is actually GROUNDED in retrieved chunks, not just factually correct
- Wiring a regression gate (notebook 33's exact pattern) so a chunking/retrieval config change can't silently regress retrieval quality

## Why This Matters
Notebooks 14/15 and the `07_rag_learning/` series taught you how to BUILD a RAG pipeline. This capstone teaches you how to know whether it's actually working, and how to keep it working as you tune chunk size, `top_k`, or the embedding model — the exact gap between "I built a RAG demo" and "I run a RAG system in production" that most tutorials skip.


In [ ]:
import sys, os
sys.path.insert(0, "..")   # so `import ragkit` works from 06_projects/

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
TEACH_MODEL = "claude-haiku-4-5-20251001"

if HAS_ANTHROPIC:
    import anthropic
    client = anthropic.Anthropic()
    print("Anthropic ready.")
else:
    client = None
    print("No ANTHROPIC_API_KEY — live generation/judging cells will be skipped.")

def ask(prompt, system="Answer directly.", max_tokens=200, temperature=0.0):
    if not HAS_ANTHROPIC:
        return "[skipped: no ANTHROPIC_API_KEY]"
    msg = client.messages.create(model=TEACH_MODEL, max_tokens=max_tokens, system=system,
                                  temperature=temperature, messages=[{"role": "user", "content": prompt}])
    return msg.content[0].text


## Assembling the pipeline from `ragkit`

Every piece here already exists in `ragkit` (built for notebooks 14/15 and the `07_rag_learning/` series) — this cell's job is composition, not new implementation. That's deliberate: a production pipeline function should be a thin, testable wrapper around well-factored pieces, not a monolith.

In [ ]:
from ragkit.data import load_corpus, chunk_text
from ragkit.vectorstore import build_collection, query_collection
from ragkit.llm import generate

def build_rag_index(chunk_size=200, overlap=40, persist_dir="/tmp/p2_chroma"):
    docs = load_corpus()   # every .txt in data/corpus/ — the Helios Robotics handbook
    texts, metas = [], []
    for doc in docs:
        for i, piece in enumerate(chunk_text(doc["text"], chunk_size=chunk_size, overlap=overlap)):
            texts.append(piece)
            metas.append({"source": doc["source"], "category": doc["category"], "chunk": i})
    collection = build_collection("p2_rag", texts, metas, persist_dir=persist_dir, reset=True)
    return collection

def rag_answer(collection, question, top_k=3):
    hits = query_collection(collection, question, k=top_k)
    context = "\n\n".join(f"[{h.metadata.get('source', '?')}] {h.text}" for h in hits)
    prompt = (f"Answer the question using ONLY the context below. If the context doesn't "
              f"contain the answer, say so explicitly.\n\nContext:\n{context}\n\nQuestion: {question}")
    answer = generate(prompt, system="You are a precise technical support assistant.")
    return answer, hits

collection = build_rag_index()
print(f"Indexed collection with {collection.count()} chunks.")


In [ ]:
answer, hits = rag_answer(collection, "What is the part number of the HR-EE-GRIP-01 gripper?")
print("Answer:", answer)
print("\nRetrieved from:", [h.metadata.get("source") for h in hits])


## Retrieval evals: hit-rate@k and MRR

Notebook 24 built an eval harness for Q&A OUTPUT quality. Retrieval has its own, earlier failure mode: the right chunk might never even reach the LLM. A golden set here maps each question to the SOURCE DOCUMENT that should be retrieved — this scores the retriever in isolation, before the generation step can mask a retrieval bug by getting lucky on partial context.

In [ ]:
RETRIEVAL_GOLDEN = [
    {"q": "What is the part number of the HR-EE-GRIP-01 gripper?", "expected_source": "spec_gripper_ee01.txt"},
    {"q": "What caused the reed switch false open signal in the gripper incident?", "expected_source": "incident_inc2024011.txt"},
    {"q": "What battery pack is used when replacing the HeliosBase M1 battery?", "expected_source": "proc_battery_replacement.txt"},
    {"q": "What is the maximum payload of the HeliosArm V2?", "expected_source": "faq_general.txt"},
]

def hit_rate_at_k(collection, golden, k=3):
    hits_count = 0
    for item in golden:
        results = query_collection(collection, item["q"], k=k)
        sources = {h.metadata.get("source") for h in results}
        if item["expected_source"] in sources:
            hits_count += 1
    return hits_count / len(golden)

def mean_reciprocal_rank(collection, golden, k=5):
    reciprocal_ranks = []
    for item in golden:
        results = query_collection(collection, item["q"], k=k)
        rank = next((h.rank for h in results if h.metadata.get("source") == item["expected_source"]), None)
        reciprocal_ranks.append(1.0 / rank if rank else 0.0)
    return sum(reciprocal_ranks) / len(reciprocal_ranks)

hr = hit_rate_at_k(collection, RETRIEVAL_GOLDEN, k=3)
mrr = mean_reciprocal_rank(collection, RETRIEVAL_GOLDEN, k=5)
print(f"Hit-rate@3: {hr:.0%}")
print(f"MRR@5:      {mrr:.2f}")


## Faithfulness judging: grounded, not just correct

An answer can be factually correct while NOT being grounded in what was retrieved (the model used background knowledge instead of the context — a hallucination risk notebook 14 demonstrated). Faithfulness judging, reusing notebook 26's model-graded pattern, scores whether every claim in the answer traces back to the retrieved chunks specifically — a different question from "is the answer right," and the one that actually matters for a RAG system's core promise.

In [ ]:
FAITHFULNESS_JUDGE_SYSTEM = (
    "You are grading whether an ANSWER is faithfully grounded in the given CONTEXT. Reply with "
    "ONLY 'GROUNDED' if every claim in the answer is supported by the context, or 'UNGROUNDED' "
    "if the answer states anything the context doesn't support. No explanation."
)

def faithfulness_score(question, context, answer):
    if not HAS_ANTHROPIC:
        return None
    prompt = f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer: {answer}\n\nVerdict:"
    verdict = ask(prompt, system=FAITHFULNESS_JUDGE_SYSTEM, max_tokens=10)
    return "GROUNDED" in verdict.upper()

def evaluate_faithfulness(collection, golden, top_k=3):
    results = []
    for item in golden:
        answer, hits = rag_answer(collection, item["q"], top_k=top_k)
        context = "\n\n".join(h.text for h in hits)
        grounded = faithfulness_score(item["q"], context, answer)
        results.append({"q": item["q"], "grounded": grounded})
    return results

faithfulness_results = evaluate_faithfulness(collection, RETRIEVAL_GOLDEN)
for r in faithfulness_results:
    print(f"[{'GROUNDED' if r['grounded'] else 'UNGROUNDED'}] {r['q']}")
if HAS_ANTHROPIC:
    faithfulness_rate = sum(1 for r in faithfulness_results if r["grounded"]) / len(faithfulness_results)
    print(f"\nFaithfulness rate: {faithfulness_rate:.0%}")


## A regression gate for retrieval config changes

Reusing notebook 33's exact `regression_gate` shape: instead of comparing two prompt files, compare two retrieval configurations (e.g. different `chunk_size`) on the SAME frozen golden set, and block the change if hit-rate regresses. This is the piece that makes "let's try smaller chunks" a measured decision instead of a guess.

In [ ]:
def retrieval_regression_gate(current_config, candidate_config, golden, tolerance=0.0):
    """Returns (passed, current_hr, candidate_hr). Same shape as notebook 33's regression_gate,
    applied to retrieval config instead of a prompt file."""
    current_collection = build_rag_index(**current_config, persist_dir="/tmp/p2_current")
    candidate_collection = build_rag_index(**candidate_config, persist_dir="/tmp/p2_candidate")
    current_hr = hit_rate_at_k(current_collection, golden, k=3)
    candidate_hr = hit_rate_at_k(candidate_collection, golden, k=3)
    passed = candidate_hr >= current_hr - tolerance
    return passed, current_hr, candidate_hr

CURRENT_CONFIG = {"chunk_size": 200, "overlap": 40}
CANDIDATE_CONFIG = {"chunk_size": 80, "overlap": 10}   # much smaller chunks — a real config change

passed, current_hr, candidate_hr = retrieval_regression_gate(CURRENT_CONFIG, CANDIDATE_CONFIG, RETRIEVAL_GOLDEN)
print(f"current chunk_size=200 hit-rate:   {current_hr:.0%}")
print(f"candidate chunk_size=80 hit-rate:  {candidate_hr:.0%}")
print(f"GATE {'PASSED — safe to adopt smaller chunks' if passed else 'FAILED — smaller chunks regress retrieval'}")


## Exercises

**Exercise 1 (Warm-up):** Add a 5th item to `RETRIEVAL_GOLDEN` targeting a document you haven't queried yet (check `data/corpus/` for options), and confirm `hit_rate_at_k` picks it up correctly.

**Exercise 2 (Apply):** Implement `precision_at_k(collection, golden, k)` — the fraction of the top-k retrieved chunks per query that come from the EXPECTED source, averaged across the golden set (a stricter metric than hit-rate, which only checks presence anywhere in the top-k).

**Exercise 3 (Extend):** Notebook 02's `07_rag_learning/02_retrieve_and_rerank.ipynb` added a cross-encoder reranking stage after initial retrieval. Sketch how you'd insert `ragkit.rerank` into `rag_answer` here, and which of this notebook's metrics (hit-rate, MRR, faithfulness) you'd expect reranking to move the most.


In [ ]:
# Exercise 1: Warm-up
# Task: Add a 5th (question, expected_source) pair from an unused data/corpus/ file, re-run hit_rate_at_k.
# Hint: ls ../data/corpus/ for filenames; pick a question whose answer is clearly IN that file.

# YOUR CODE HERE


# Exercise 2: Apply
# Task: Implement precision_at_k(collection, golden, k) -> float (fraction of top-k FROM the
# expected source, averaged across golden).
# Hint: for each query, count how many of the k retrieved hits match expected_source, divide by k.

# YOUR CODE HERE


# Exercise 3: Extend
# Task: Sketch adding ragkit.rerank into rag_answer, and predict which metric moves most.
# Hint: reranking re-orders an already-retrieved candidate set — it can improve MRR (better
# ranking) without necessarily changing hit-rate (same candidates, different order).

# YOUR CODE HERE


<details>
<summary>Click to reveal solutions</summary>

```python
# Exercise 1
RETRIEVAL_GOLDEN_EXT = RETRIEVAL_GOLDEN + [
    {"q": "What is included in the arm commissioning procedure?", "expected_source": "proc_arm_commissioning.txt"},
]
print(hit_rate_at_k(collection, RETRIEVAL_GOLDEN_EXT, k=3))

# Exercise 2
def precision_at_k(collection, golden, k=3):
    precisions = []
    for item in golden:
        results = query_collection(collection, item["q"], k=k)
        matches = sum(1 for h in results if h.metadata.get("source") == item["expected_source"])
        precisions.append(matches / k)
    return sum(precisions) / len(precisions)

print(precision_at_k(collection, RETRIEVAL_GOLDEN, k=3))

# Exercise 3
from ragkit.rerank import rerank   # same cross-encoder used in 07_rag_learning/02

def rag_answer_reranked(collection, question, top_k=3, candidate_k=10):
    candidates = query_collection(collection, question, k=candidate_k)
    reranked = rerank(question, candidates, top_k=top_k)   # cross-encoder re-scores AND truncates
    context = "\n\n".join(f"[{h.metadata.get('source', '?')}] {h.text}" for h in reranked)
    prompt = f"Answer using ONLY this context:\n{context}\n\nQuestion: {question}"
    return generate(prompt), reranked
# Expect MRR to improve the most — reranking reorders the SAME candidate pool retrieved by the
# bi-encoder, so hit-rate (is the right doc anywhere in top-k) barely changes, but ranking the
# right document HIGHER within that pool directly raises reciprocal rank.
```
</details>

## Key Takeaways
- A production RAG pipeline is composition, not new implementation — `ragkit`'s existing chunk/embed/store/retrieve/generate pieces, wired into one thin, testable function.
- Retrieval evals (hit-rate@k, MRR) score the RETRIEVER in isolation — a generation step that happens to produce a good answer can mask a retrieval bug that will bite on a harder question.
- Faithfulness judging (notebook 26's pattern) asks a different question than correctness: is every claim actually traceable to the retrieved context, not just true in general.
- A regression gate (notebook 33's exact shape) turns "let's try smaller chunks" from a guess into a measured, blockable decision — apply this to ANY retrieval config change before shipping it.
- This is the gap between "I built a RAG demo" (Tier 3, `07_rag_learning/`) and "I run a RAG system in production" — the evaluation and gating discipline, not new retrieval tricks.

## What's Next
P4 applies this same evaluation discipline to a multi-agent system — extending P3's research agent with the security, structured-output, and agent-eval tooling built across Tiers 8 and 5.
